In [ ]:
import torch
import datasets
from datasets import load_dataset

from torchvision import transforms, datasets

In [ ]:
from google.colab import userdata
#hf_token=userdata.get('proj2_Plant_Disease_Classifier')

In [ ]:
dataset= load_dataset("mohanty/PlantVillage", "default")

In [ ]:
type(dataset)

In [ ]:
# Training transforms — with augmentation
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),           # slightly larger than 224
    transforms.RandomCrop(224),              # random crop to 224 — better than direct resize
    transforms.RandomHorizontalFlip(),       # flip left/right
    transforms.RandomVerticalFlip(),         # flip upside down (valid for leaves)
    transforms.RandomRotation(15),           # rotate up to 15 degrees
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,                      # slight colour variation
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Test transforms — NO augmentation, just resize and normalize
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
class PlantDiseaseDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"].convert("RGB")  # ensure 3 channels
        label = item["label"]
        if self.transform:
            image = self.transform(image)
        return image, label

# Step 5 — create train and test datasets
train_dataset = PlantDiseaseDataset(dataset["train"], transform=train_transform)
test_dataset  = PlantDiseaseDataset(dataset["test"],  transform=test_transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

In [ ]:
images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)   # should be [32, 3, 224, 224]
print("Batch label shape:", labels.shape)   # should be [32]

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import time

class DiseaseClassifier:

    def __init__(self, num_classes, device=None):
        # Step 1 — detect device (GPU if available, else CPU)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Step 2 — load pretrained EfficientNet-B0
        self.model = models.efficientnet_b0(weights="IMAGENET1K_V1")

        # Step 3 — freeze all layers (keep ImageNet knowledge intact)
        for param in self.model.parameters():
            param.requires_grad = False

        # Step 4 — replace final classifier layer for 38 classes
        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, num_classes)

        # Step 5 — move model to GPU
        self.model = self.model.to(self.device)

        # Step 6 — loss function and optimizer
        # only train the new classifier layer parameters
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(
            self.model.classifier.parameters(), lr=0.001
        )

        # Step 7 — learning rate scheduler
        # reduces LR by 0.1 if val loss doesn't improve for 3 epochs
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", patience=3, factor=0.1
        )

        # Step 8 — tracking
        self.history = {"train_loss": [], "train_acc": [],
                        "val_loss":   [], "val_acc":   []}

    def train_one_epoch(self, train_loader):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.to(self.device)
            labels = labels.to(self.device)

            self.optimizer.zero_grad()        # clear previous gradients
            outputs = self.model(images)      # forward pass
            loss = self.criterion(outputs, labels)  # compute loss
            loss.backward()                   # backpropagation
            self.optimizer.step()             # update weights

            running_loss += loss.item()
            _, predicted = outputs.max(1)     # get highest confidence class
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            # print progress every 50 batches
            if (batch_idx + 1) % 50 == 0:
                print(f"  Batch {batch_idx+1}/{len(train_loader)} "
                      f"Loss: {running_loss/(batch_idx+1):.4f} "
                      f"Acc: {100.*correct/total:.2f}%")

        epoch_loss = running_loss / len(train_loader)
        epoch_acc  = 100. * correct / total
        return epoch_loss, epoch_acc

    def evaluate(self, test_loader):
        self.model.eval()
        running_loss, correct, total = 0.0, 0, 0

        with torch.no_grad():               # no gradient calculation needed
            for images, labels in test_loader:
                images = images.to(self.device)
                labels = labels.to(self.device)

                outputs = self.model(images)
                loss    = self.criterion(outputs, labels)

                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total   += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(test_loader)
        epoch_acc  = 100. * correct / total
        return epoch_loss, epoch_acc

    def train(self, train_loader, test_loader, epochs=10):
        print(f"Starting training for {epochs} epochs...\n")
        best_val_acc = 0.0

        for epoch in range(epochs):
            start = time.time()
            print(f"Epoch {epoch+1}/{epochs}")
            print("-" * 40)

            # train
            train_loss, train_acc = self.train_one_epoch(train_loader)

            # evaluate
            val_loss, val_acc = self.evaluate(test_loader)

            # step scheduler
            self.scheduler.step(val_loss)

            # record history
            self.history["train_loss"].append(train_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_loss"].append(val_loss)
            self.history["val_acc"].append(val_acc)

            elapsed = time.time() - start
            print(f"\nTrain Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%")
            print(f"Val   Loss: {val_loss:.4f}  Val   Acc: {val_acc:.2f}%")
            print(f"Time: {elapsed:.1f}s")

            # save best model
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                self.save("best_model.pt")
                print(f"  ✓ Best model saved (val acc: {val_acc:.2f}%)")

            print()

        print(f"Training complete. Best val accuracy: {best_val_acc:.2f}%")

    def save(self, path):
        torch.save({
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "history": self.history
        }, path)

    def load(self, path):
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.history = checkpoint["history"]
        print(f"Model loaded from {path}")

    def predict(self, image, class_names):
        """
        image       : a PIL Image object
        class_names : list from dataset["train"].features["label"].names
        """
        self.model.eval()

        # Step 1 — apply same transforms as test set (NO augmentation)
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
        ])

        # Step 2 — prepare image tensor
        image_tensor = transform(image.convert("RGB"))  # [3, 224, 224]
        image_tensor = image_tensor.unsqueeze(0)        # [1, 3, 224, 224] — add batch dimension
        image_tensor = image_tensor.to(self.device)

        # Step 3 — run through model
        with torch.no_grad():
            outputs = self.model(image_tensor)          # raw scores [1, 38]
            probs   = torch.softmax(outputs, dim=1)     # convert to probabilities

            # top 3 predictions
            top3_probs, top3_indices = torch.topk(probs, k=3, dim=1)

        # Step 4 — format results
        results = []
        for i in range(3):
            idx        = top3_indices[0][i].item()
            confidence = top3_probs[0][i].item() * 100
            disease    = class_names[idx].replace("___", " — ").replace("_", " ")
            results.append({
                "rank":       i + 1,
                "disease":    disease,
                "confidence": round(confidence, 2),
                "raw_label":  class_names[idx]
            })

        return results


# ── Run it ──────────────────────────────────────────────
num_classes = len(dataset["train"].features["label"].names)

classifier = DiseaseClassifier(num_classes=num_classes)
classifier.train(train_loader, test_loader, epochs=10)

In [ ]:
from PIL import Image
import requests
from io import BytesIO

# grab a sample image from your test set
sample      = dataset["test"][0]
sample_image = sample["image"]
true_label  = class_names[sample["label"]]

# get class names
class_names = dataset["train"].features["label"].names

# run prediction
predictions = classifier.predict(sample_image, class_names)

print(f"True label : {true_label}")
print(f"\nTop 3 predictions:")
for p in predictions:
    print(f"  #{p['rank']} {p['disease']:<40} {p['confidence']}%")